In [ ]:
# -*- coding: utf-8 -*-
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. 이미지 로드
def load_image(path):
    img = sitk.ReadImage(path)
    print(f"Original spacing: {img.GetSpacing()}")
    print(f"Original size: {img.GetSize()}")
    return img

# 2. 리샘플링
def resample_image(image, target_spacing=(0.5, 0.5, 0.5), interpolator=sitk.sitkLinear):
    original_spacing = image.GetSpacing()
    original_size = image.GetSize()

    new_size = [
        int(round(osz * ospc / tspc))
        for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetInterpolator(interpolator)
    
    return resampler.Execute(image)

# 3. Min–Max 정규화
def minmax_normalize(image, new_min=0.0, new_max=1.0):
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    old_min, old_max = arr.min(), arr.max()
    if old_max > old_min:
        normed = (arr - old_min) / (old_max - old_min)
        normed = normed * (new_max - new_min) + new_min
    else:
        normed = np.zeros_like(arr) + new_min

    out = sitk.GetImageFromArray(normed.astype(np.float32))
    out.SetSpacing(image.GetSpacing())
    out.SetDirection(image.GetDirection())
    out.SetOrigin(image.GetOrigin())
    return out

# 4. 중앙 패딩 및 크롭 (패딩·크롭 통합)
def pad_crop_to_target_size(image, target_size=(256,256,128)):
    arr = sitk.GetArrayFromImage(image)  # (z, y, x)
    cz, cy, cx = arr.shape
    tx, ty, tz = target_size

    # 중앙 크롭을 위한 시작/끝 인덱스 계산
    z_start = max((cz - tz)//2, 0)
    y_start = max((cy - ty)//2, 0)
    x_start = max((cx - tx)//2, 0)
    z_end   = z_start + min(tz, cz)
    y_end   = y_start + min(ty, cy)
    x_end   = x_start + min(tx, cx)

    # 크롭
    cropped = arr[z_start:z_end, y_start:y_end, x_start:x_end]

    # 패딩 크기 계산
    pad_z = max(tz - cropped.shape[0], 0)
    pad_y = max(ty - cropped.shape[1], 0)
    pad_x = max(tx - cropped.shape[2], 0)
    before = (pad_z//2, pad_y//2, pad_x//2)
    after  = (pad_z - before[0], pad_y - before[1], pad_x - before[2])

    padded = np.pad(
        cropped,
        ( (before[0], after[0]),
          (before[1], after[1]),
          (before[2], after[2]) ),
        mode='constant', constant_values=0
    )

    out = sitk.GetImageFromArray(padded)
    out.SetSpacing(image.GetSpacing())
    out.SetDirection(image.GetDirection())
    out.SetOrigin(image.GetOrigin())
    return out


# 5. 시각화 함수 
def plot_preprocessed_image(original_image, processed_image, title="Original vs Preprocessed Image"):
    orig = sitk.GetArrayFromImage(original_image)
    proc = sitk.GetArrayFromImage(processed_image)

    # 중앙 slice
    o_cz, o_cy, o_cx = orig.shape[0]//2, orig.shape[1]//2, orig.shape[2]//2
    p_cz, p_cy, p_cx = proc.shape[0]//2, proc.shape[1]//2, proc.shape[2]//2

    slices = [
        (orig[o_cz], proc[p_cz], "Axial"),
        (orig[:, o_cy, :].T, proc[:, p_cy, :].T, "Coronal"),
        (orig[:, :, o_cx].T, proc[:, :, p_cx].T, "Sagittal"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    plt.suptitle(title)
    for i, (o_sl, p_sl, name) in enumerate(slices):
        axes[0, i].imshow(o_sl, cmap="gray")
        axes[0, i].set_title(f"Orig {name}")
        axes[1, i].imshow(p_sl, cmap="gray")
        axes[1, i].set_title(f"Proc {name}")
        axes[0, i].axis("off"); axes[1, i].axis("off")
    plt.tight_layout()
    plt.show()

# 6. 파이프라인 (정규화 → 패딩·크롭 순서)
def preprocess_image(path,
                     target_spacing=(0.5, 0.5, 0.5),
                     target_size=(256, 256, 128),
                     norm_range=(0.0, 1.0)):
    image = load_image(path)

    print("→ Resampling to uniform spacing...")
    resampled = resample_image(image, target_spacing=target_spacing)

    print("→ Applying min-max normalization...")
    normalized = minmax_normalize(resampled, new_min=norm_range[0], new_max=norm_range[1])

    print("→ Padding/Cropping to target size...")
    final = pad_crop_to_target_size(normalized, target_size=target_size)

    print(f"Final size:    {final.GetSize()}")
    print(f"Final spacing: {final.GetSpacing()}")
    return final

# 7. 이미지 저장
def save_image(image, path):
    sitk.WriteImage(image, path)
    print(f"Saved image to {path}")

In [ ]:
input_path = '/public/sylee/MDAISS_2/output/nifit/EUMC_S3_00003_S3_00003_TOF_grayscale.nii.gz'
output_path = '/public/sylee/MDAISS/output/pre/preprocessed_image.nii.gz'

orig = load_image(input_path)
result = preprocess_image(input_path, target_spacing=(1.5, 1.5, 1.5), target_size=(128, 128, 128))
#result = preprocess_image(input_path, target_spacing=(0.75, 0.75, 0.75), target_size=(256, 256, 256))
plot_preprocessed_image(orig, result, title="Original vs Preprocessed")

save_image(result, output_path)

In [ ]:
import os
import glob

input_dir= '/public/sylee/MDAISS_2/output/nifit/'
output_dir= '/public/sylee/MDAISS_2/output/processed_V2/'

os.makedirs(output_dir, exist_ok=True)
file_list = glob.glob(os.path.join(input_dir, '*.nii.gz'))

for idx, file_path in enumerate(file_list):
    
    result = preprocess_image(
        file_path,
        target_spacing=(0.75, 0.75, 0.75), 
        target_size=(256, 256, 256)
    )
    
    base_name = os.path.basename(file_path).replace('.nii.gz', '_processed.nii.gz')
    out_path = os.path.join(output_dir, base_name)
    save_image(result, out_path)
    print(f"Saved: {out_path}")
